<a href="https://colab.research.google.com/github/YathinKandakatla/DataManagement/blob/main/PS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PS3: Data Management and Merging
## County-Level Excessive Drinking and Economic Conditions in New Jersey, 2014-2023

# Research Perspective

## Research Abstract
This project continues the alcohol research in PS1 and PS2 by examining New Jersey counties across the ten-year study period 2014-2023. The main dataset contains county excessive-drinking estimates. Census income and poverty data are added first, followed by BLS unemployment data. Both merges use county FIPS and the actual data year.

The final dataset retains 21 counties in each of ten years: 210 county-year observations. Alcohol estimates are available for nine of the ten observation years. New Jersey's 2019 alcohol values are unavailable; those county-year rows remain in the panel with missing alcohol values. No 2023 estimate or historical average is substituted for earlier observations.

The alcohol measure is excessive drinking (binge or heavy drinking), which is broader than the binge drinking measure in PS2. The change provides longer county-level coverage and is stated explicitly. Historical health estimates are modeled snapshots with changes in methodology, so the project studies county differences and within-year economic associations, rather than claiming a comparable ten-year drinking trend.

## Research Questions
1. How do county excessive-drinking estimates differ within each available observation year?
2. Within the same year, are counties with higher poverty or unemployment also estimated to have higher excessive drinking?
3. What patterns emerge after county alcohol, income/poverty and unemployment are merged across all ten study years?
4. How did observed county poverty and unemployment vary over 2014-2023, providing context for the alcohol comparisons?

## Motivation
Statewide summaries can hide differences between counties. County-year data allow economic conditions and alcohol estimates to be aligned to the same geography and observation year, instead of comparing a single health year with a historical economic average.

The goal is to build one coherent dataset for describing alcohol and local economic context. Observed economic time series and modeled alcohol snapshots are interpreted according to their different strengths and limitations.

## Hypotheses
County alcohol estimates may be associated with economic conditions, but the direction is uncertain. Higher poverty or unemployment could coincide with greater hardship, while higher income could also coincide with different drinking patterns. The analysis examines these associations within each observation year and does not assume a causal effect.

# Unit of Analysis
The unit is a **New Jersey county-year**, covering 21 counties and 2014-2023.

The merge keys are **County_FIPS and Year**. Year is the alcohol observation year and the economic observation year, not the health publication/release year. The final panel has 210 rows; 189 have alcohol estimates and 21 rows for 2019 have missing alcohol estimates.

Income refers to households, poverty refers to people in the poverty universe, unemployment refers to the civilian labor force, and alcohol prevalence refers to adults. These different denominators prevent individual-level interpretations.

# Datasets

## Dataset 1: County Excessive Drinking in New Jersey
**Source:** County Health Rankings & Roadmaps, using CDC BRFSS modeled estimates. **Format:** CSV.

The main variable is the percentage of adults reporting binge or heavy drinking. We keep the estimate and its 95% interval. Raw proportions are multiplied by 100 to express percentages.

Historical archive: https://www.countyhealthrankings.org/health-data/methodology-and-sources/data-documentation/national-data-documentation-2010-2023


## Dataset 2: County Income and Poverty
**Source:** U.S. Census Bureau SAIPE. **Years:** 2014-2023. **Format:** fixed-width text.

The variables are annual all-age poverty percentage and annual median household income. This dataset supplies economic context for the same county and year as the alcohol estimate.

Source: https://www.census.gov/data/datasets/2023/demo/saipe/2023-state-and-county.html


## Dataset 3: County Unemployment
**Source:** BLS Local Area Unemployment Statistics, accessed directly through the public BLS API while FRED is under maintenance. **Years:** 2014-2023. **Format:** JSON converted to a pandas table.

Each retained value is the published annual average (period M13) for that county and observation year.

Original source: https://www.bls.gov/lau/tables.htm
API documentation: https://www.bls.gov/developers/api_signature.htm


# Merge Plan
**Merge 1:** Alcohol county-year observations + Census county-year observations, using County_FIPS and Year.

**Merge 2:** The result of Merge 1 + BLS county-year unemployment, using the same two keys.

No prior economic averages are used. No merge is restricted to 2023. Outer merges preserve the full ten-year scope and reveal unmatched keys. Matching, duplicate keys, missing values and coverage are investigated after each merge.

# Data Analysis

## 1. Import Packages

In [1]:
# I import the tools I need to download data, organize tables, and create charts.
import pandas as pd
import plotly.express as px
import requests
from io import StringIO


In [2]:
# I set my ten study years and use county FIPS and year as the matching keys.
study_years = list(range(2014, 2024))
merge_keys = ["County_FIPS", "Year"]
study_years

[2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

## 2. Load Dataset 1: Alcohol

### 2.1 Load and Inspect One Source File
Download the source text with requests, read it with pandas, and inspect the first rows. The same approach is used for the other source files.


In [3]:
# I download one alcohol file and preview it to understand its structure.
alcohol_url = "https://www.countyhealthrankings.org/sites/default/files/analytic_data2016.csv"
download = requests.get(alcohol_url, timeout=90)
download.raise_for_status()
alcohol_example = pd.read_csv(StringIO(download.text), skiprows=1)
alcohol_example.head()


,statecode,countycode,fipscode,state,county,year,county_ranked,v001_rawvalue,v001_numerator,v001_denominator,...,v013_rawvalue,v016_rawvalue,v017_rawvalue,v025_rawvalue,v026_rawvalue,v027_rawvalue,v031_rawvalue,v032_rawvalue,v035_rawvalue,v097_rawvalue
0,0,0,0,US,United States,2016,NaN,6605.3,3362601.0,884075121.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,0,1000,AL,Alabama,2016,NaN,9544.7,73929.0,13566808.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1,1,1001,AL,Autauga County,2016,1.0,9215.1,809.0,157582.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1,3,1003,AL,Baldwin County,2016,1.0,7454.6,2506.0,531694.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1,5,1005,AL,Barbour County,2016,1.0,8499.9,410.0,76276.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# I check the raw alcohol file's rows and columns before cleaning it.
alcohol_example.shape

(3193, 482)

In [5]:
# I inspect the column names to find the alcohol measures and county identifiers.
alcohol_example.columns

Index(['statecode', 'countycode', 'fipscode', 'state', 'county', 'year',
       'county_ranked', 'v001_rawvalue', 'v001_numerator', 'v001_denominator',
       ...
       'v013_rawvalue', 'v016_rawvalue', 'v017_rawvalue', 'v025_rawvalue',
       'v026_rawvalue', 'v027_rawvalue', 'v031_rawvalue', 'v032_rawvalue',
       'v035_rawvalue', 'v097_rawvalue'],
      dtype='object', length=482)

### 2.2 Load the Other Observation Years

In [6]:
# I link each alcohol observation year to its correct source file.
alcohol_urls = {
    2014: "https://www.countyhealthrankings.org/sites/default/files/analytic_data2016.csv",
    2015: "https://www.countyhealthrankings.org/sites/default/files/analytic_data2017.csv",
    2016: "https://www.countyhealthrankings.org/sites/default/files/analytic_data2018_0.csv",
    2017: "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2020_0.csv",
    2018: "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2021.csv",
    2020: "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2023_0.csv",
    2021: "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2024.csv",
    2022: "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025_v3.csv"
}

In [7]:
# I choose the county identifiers, alcohol estimate, and confidence limits I need.
alcohol_columns = ["state", "countycode", "fipscode", "county",
                   "v049_rawvalue", "v049_cilow", "v049_cihigh"]

In [8]:
# I load the yearly alcohol files and stack their rows into one dataset.
alcohol_frames = []
for year, url in alcohol_urls.items():
    download = requests.get(url, timeout=90)
    download.raise_for_status()
    data = pd.read_csv(StringIO(download.text), skiprows=1, usecols=alcohol_columns)
    data["Year"] = year
    alcohol_frames.append(data)

alcohol_raw = pd.concat(alcohol_frames, ignore_index=True)
alcohol_raw.head()


,countycode,fipscode,state,county,v049_rawvalue,v049_cilow,v049_cihigh,Year
0,0,0.0,US,United States,NaN,NaN,NaN,2014
1,0,1000.0,AL,Alabama,0.133,0.122,0.145,2014
2,1,1001.0,AL,Autauga County,0.151,0.145,0.157,2014
3,3,1003.0,AL,Baldwin County,0.151,0.146,0.157,2014
4,5,1005.0,AL,Barbour County,0.124,0.119,0.128,2014


In [9]:
# I add the 2023 alcohol data separately because its file uses a different header format.
# The latest supplement contains 2023 observations and has one header row.
alcohol_2023_url = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_supplement_20260325%5B1%5D.csv"
download = requests.get(alcohol_2023_url, timeout=90)
download.raise_for_status()
alcohol_2023 = pd.read_csv(StringIO(download.text), usecols=alcohol_columns)
alcohol_2023["Year"] = 2023
alcohol_raw = pd.concat([alcohol_raw, alcohol_2023], ignore_index=True)


### 2.3 Keep New Jersey Counties and Rename Variables

In [10]:
# I keep only New Jersey counties and exclude the state totals.
alcohol = alcohol_raw.loc[
    (alcohol_raw["state"] == "NJ") & (alcohol_raw["countycode"] != 0)
].copy()
alcohol.head()

,countycode,fipscode,state,county,v049_rawvalue,v049_cilow,v049_cihigh,Year
1804,1,34001.0,NJ,Atlantic County,0.177,0.172,0.181,2014
1805,3,34003.0,NJ,Bergen County,0.176,0.172,0.180,2014
1806,5,34005.0,NJ,Burlington County,0.194,0.189,0.199,2014
1807,7,34007.0,NJ,Camden County,0.177,0.172,0.182,2014
1808,9,34009.0,NJ,Cape May County,0.181,0.176,0.187,2014


In [11]:
# I keep only the alcohol variables and identifiers needed for my analysis.
alcohol = alcohol[["Year", "fipscode", "county", "v049_rawvalue", "v049_cilow", "v049_cihigh"]].copy()
alcohol.head()

,Year,fipscode,county,v049_rawvalue,v049_cilow,v049_cihigh
1804,2014,34001.0,Atlantic County,0.177,0.172,0.181
1805,2014,34003.0,Bergen County,0.176,0.172,0.180
1806,2014,34005.0,Burlington County,0.194,0.189,0.199
1807,2014,34007.0,Camden County,0.177,0.172,0.182
1808,2014,34009.0,Cape May County,0.181,0.176,0.187


In [12]:
# I rename the alcohol columns so their meanings are easier to understand.
alcohol = alcohol.rename(columns={
    "fipscode": "County_FIPS",
    "county": "County",
    "v049_rawvalue": "Alcohol_Prevalence",
    "v049_cilow": "Alcohol_CI_Lower",
    "v049_cihigh": "Alcohol_CI_Upper"
})
alcohol.head()

,Year,County_FIPS,County,Alcohol_Prevalence,Alcohol_CI_Lower,Alcohol_CI_Upper
1804,2014,34001.0,Atlantic County,0.177,0.172,0.181
1805,2014,34003.0,Bergen County,0.176,0.172,0.180
1806,2014,34005.0,Burlington County,0.194,0.189,0.199
1807,2014,34007.0,Camden County,0.177,0.172,0.182
1808,2014,34009.0,Cape May County,0.181,0.176,0.187


In [13]:
# I standardize county codes and convert alcohol estimates and confidence limits to percentages.
alcohol["County_FIPS"] = alcohol["County_FIPS"].astype(int).astype(str).str.zfill(5)
alcohol["Alcohol_Prevalence"] = alcohol["Alcohol_Prevalence"] * 100
alcohol["Alcohol_CI_Lower"] = alcohol["Alcohol_CI_Lower"] * 100
alcohol["Alcohol_CI_Upper"] = alcohol["Alcohol_CI_Upper"] * 100
alcohol.dtypes

,0
Year,int64
County_FIPS,object
County,object
Alcohol_Prevalence,float64
Alcohol_CI_Lower,float64
Alcohol_CI_Upper,float64


In [14]:
# I count counties by year to check alcohol coverage and identify the missing 2019 year.
alcohol["Year"].value_counts().sort_index()

,count
Year,
2014,21
2015,21
2016,21
2017,21
2018,21
2020,21
2021,21
2022,21
2023,21


In [15]:
# I check for repeated alcohol county-year records before merging.
alcohol.duplicated(subset=merge_keys).sum()

np.int64(0)

In [16]:
# I check the loaded alcohol records for missing values.
alcohol.isna().sum()

,0
Year,0
County_FIPS,0
County,0
Alcohol_Prevalence,0
Alcohol_CI_Lower,0
Alcohol_CI_Upper,0


### Interpretation of Alcohol Coverage
There are 21 county records in each of nine actual alcohol observation years, or 189 rows. The only absent study year is 2019. It is not filled with the archive's carried-forward 2018 estimates. The subsequent outer merge retains all ten economic observation years and exposes the missing alcohol year explicitly.

## 3. Load Dataset 2: Census Income and Poverty

### 3.1 Read One Year and Inspect It

In [17]:
# I define the Census fields and their text positions so I can read the fixed-width file.
census_url = "https://www2.census.gov/programs-surveys/saipe/datasets/2023/2023-state-and-county/est23-nj.txt"
census_columns = ["State_FIPS", "County_Code", "Poverty_Percent", "Household_Income", "Census_County"]
census_positions = [(0, 2), (3, 6), (34, 38), (133, 139), (193, 238)]

In [18]:
# I preview one Census file to check that its fields are read correctly.
# The raw Census file is fixed-width text, so use read_fwf rather than read_csv.
download = requests.get(census_url, timeout=90)
download.raise_for_status()
census_example = pd.read_fwf(StringIO(download.text), colspecs=census_positions, names=census_columns)
census_example.head()


,State_FIPS,County_Code,Poverty_Percent,Household_Income,Census_County
0,34,0,9.7,99716,New Jersey
1,34,1,11.9,75904,Atlantic County
2,34,3,6.6,116157,Bergen County
3,34,5,6.8,102384,Burlington County
4,34,7,12.3,82678,Camden County


### 3.2 Load All Ten Census Years

In [19]:
# I download and stack all ten Census years to build my economic dataset.
census_frames = []
census_links = []
for year in study_years:
    suffix = str(year)[-2:]
    url = f"https://www2.census.gov/programs-surveys/saipe/datasets/{year}/{year}-state-and-county/est{suffix}-nj.txt"
    download = requests.get(url, timeout=90)
    download.raise_for_status()
    data = pd.read_fwf(StringIO(download.text), colspecs=census_positions, names=census_columns)
    data["Year"] = year
    census_frames.append(data)
    census_links.append(url)

census_raw = pd.concat(census_frames, ignore_index=True)
census_raw.head()


,State_FIPS,County_Code,Poverty_Percent,Household_Income,Census_County,Year
0,34,0,11.1,71968,New Jersey,2014
1,34,1,15.1,54208,Atlantic County,2014
2,34,3,7.6,84309,Bergen County,2014
3,34,5,7.2,80479,Burlington County,2014
4,34,7,13.0,61685,Camden County,2014


In [20]:
# I remove Census state totals and format county codes and names consistently.
# County code 0 identifies the state total, which is removed.
census = census_raw.loc[census_raw["County_Code"] != 0].copy()
census["County_FIPS"] = "34" + census["County_Code"].astype(int).astype(str).str.zfill(3)
census["Census_County"] = census["Census_County"].str.strip()

In [21]:
# I keep county, year, poverty, and income for the first merge.
census = census[["County_FIPS", "Year", "Census_County", "Poverty_Percent", "Household_Income"]].copy()
census.head()

,County_FIPS,Year,Census_County,Poverty_Percent,Household_Income
1,34001,2014,Atlantic County,15.1,54208
2,34003,2014,Bergen County,7.6,84309
3,34005,2014,Burlington County,7.2,80479
4,34007,2014,Camden County,13.0,61685
5,34009,2014,Cape May County,13.0,55409


In [22]:
# I check that Census contains the expected 210 county-year records.
census.shape

(210, 5)

In [23]:
# I check that Census identifiers and economic values have suitable data types.
census.dtypes

,0
County_FIPS,object
Year,int64
Census_County,object
Poverty_Percent,float64
Household_Income,int64


In [24]:
# I check that each Census year contains all 21 counties.
census["Year"].value_counts().sort_index()

,count
Year,
2014,21
2015,21
2016,21
2017,21
2018,21
2019,21
2020,21
2021,21
2022,21


In [25]:
# I check Census county-year keys for duplicates before merging.
census.duplicated(subset=merge_keys).sum()

np.int64(0)

In [26]:
# I check whether any Census values are missing.
census.isna().sum()

,0
County_FIPS,0
Year,0
Census_County,0
Poverty_Percent,0
Household_Income,0


## 4. Prepare for the First Merge

In [27]:
# I compare county and year coverage in alcohol and Census before joining them.
print("Alcohol county codes:", alcohol["County_FIPS"].nunique())
print("Census county codes:", census["County_FIPS"].nunique())
print("Alcohol years:", sorted(alcohol["Year"].unique()))
print("Census years:", sorted(census["Year"].unique()))

Alcohol county codes: 21
Census county codes: 21
Alcohol years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
Census years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


# Merge 1: Alcohol + Census for All Study Years

In [28]:
# I outer-merge alcohol and Census by county and year, retaining unmatched rows and checking unique keys.
alcohol_census_check = pd.merge(
    alcohol,
    census,
    how="outer",
    on=["County_FIPS", "Year"],
    indicator=True,
    validate="one_to_one"
)
alcohol_census_check.head()

,Year,County_FIPS,County,Alcohol_Prevalence,Alcohol_CI_Lower,Alcohol_CI_Upper,Census_County,Poverty_Percent,Household_Income,_merge
0,2014,34001,Atlantic County,17.700000,17.200000,18.100000,Atlantic County,15.1,54208,both
1,2015,34001,Atlantic County,17.307394,16.847267,17.757869,Atlantic County,14.1,53296,both
2,2016,34001,Atlantic County,17.411235,16.917008,17.910904,Atlantic County,14.2,55672,both
3,2017,34001,Atlantic County,17.004454,16.565697,17.468133,Atlantic County,14.4,59309,both
4,2018,34001,Atlantic County,17.185672,16.541828,17.843667,Atlantic County,12.8,60826,both


In [29]:
# I count matched and unmatched records to assess the first merge.
alcohol_census_check["_merge"].value_counts()

,count
_merge,
both,189
right_only,21
left_only,0


In [30]:
# I check match status by year to see which years contain unmatched records.
pd.crosstab(alcohol_census_check["Year"], alcohol_census_check["_merge"])

_merge,right_only,both
Year,,
2014,0,21
2015,0,21
2016,0,21
2017,0,21
2018,0,21
2019,21,0
2020,0,21
2021,0,21
2022,0,21


In [31]:
# I inspect individual county-year match statuses to verify the first merge.
alcohol_census_check[["County_FIPS", "Year", "_merge"]].sort_values(["Year", "County_FIPS"])

,County_FIPS,Year,_merge
0,34001,2014,both
10,34003,2014,both
20,34005,2014,both
30,34007,2014,both
40,34009,2014,both
...,...,...,...
169,34033,2023,both
179,34035,2023,both
189,34037,2023,both
199,34039,2023,both


In [32]:
# I inspect Census-only records to understand why they lack matching alcohol data.
census_only = alcohol_census_check.loc[
    alcohol_census_check["_merge"] == "right_only",
    ["County_FIPS", "Year", "Census_County", "Poverty_Percent", "Household_Income"]
].copy()
census_only

,County_FIPS,Year,Census_County,Poverty_Percent,Household_Income
5,34001,2019,Atlantic County,11.3,62678
15,34003,2019,Bergen County,5.7,107971
25,34005,2019,Burlington County,5.7,88443
35,34007,2019,Camden County,10.7,73168
45,34009,2019,Cape May County,9.8,66565
55,34011,2019,Cumberland County,13.8,54179
65,34013,2019,Essex County,13.8,64522
75,34015,2019,Gloucester County,7.5,87996
85,34017,2019,Hudson County,13.7,77738
95,34019,2019,Hunterdon County,4.0,117275


In [33]:
# I check the first merge's size to confirm that it retains all 210 county-years.
alcohol_census_check.shape

(210, 10)

In [34]:
# I check that the first merge did not create duplicate county-year records.
alcohol_census_check.duplicated(subset=merge_keys).sum()

np.int64(0)

In [35]:
# I check which values are missing after the first merge.
alcohol_census_check.isna().sum()

,0
Year,0
County_FIPS,0
County,21
Alcohol_Prevalence,21
Alcohol_CI_Lower,21
Alcohol_CI_Upper,21
Census_County,0
Poverty_Percent,0
Household_Income,0
_merge,0


### Explanation of Non-Merges
The verified result is **189 matched county-years and 21 Census-only county-years**, with no alcohol-only keys. All 21 Census-only rows are 2019. New Jersey lacks a new 2019 alcohol estimate, while Census has county economic observations for that year. This is an expected source coverage gap, not a county-code mismatch.

The outer merge preserves those 2019 rows. Their alcohol prevalence and intervals remain missing. No missing estimate is converted to zero or carried forward. County names for these rows come from Census.

In [36]:
# I fill missing county names from Census and remove extra columns while leaving missing alcohol values unchanged.
alcohol_census = alcohol_census_check.drop(columns="_merge").copy()
alcohol_census["County"] = alcohol_census["County"].fillna(alcohol_census["Census_County"])
alcohol_census = alcohol_census.drop(columns="Census_County")
alcohol_census["Year"].value_counts().sort_index()

,count
Year,
2014,21
2015,21
2016,21
2017,21
2018,21
2019,21
2020,21
2021,21
2022,21


## 5. Load Dataset 3: County Unemployment

### 5.1 Read One County and Inspect It
BLS is the original producer of the county unemployment data. This version requests the data directly from the public BLS API because FRED returned a maintenance webpage. One request includes all 21 counties and 2014-2023. The JSON response is converted to a table; period M13 identifies the published annual average. No local CSV upload or API key is required.


In [37]:
# I request annual BLS data for all counties and years, retry temporary 503 errors, and preview the response.
import time

unemployment_url = "https://api.bls.gov/publicAPI/v2/timeseries/data/"
series_ids = [
    f"LAUCN34{code:03d}0000000003"
    for code in range(1, 42, 2)
]

for attempt in range(3):
    download = requests.post(
        unemployment_url,
        json={
            "seriesid": series_ids,
            "startyear": "2014",
            "endyear": "2023",
            "annualaverage": True
        },
        timeout=90
    )

    if download.status_code != 503:
        break

    if attempt < 2:
        print("BLS is temporarily unavailable. Retrying in 10 seconds...")
        time.sleep(10)

download.raise_for_status()
unemployment_response = download.json()

if unemployment_response["status"] != "REQUEST_SUCCEEDED":
    raise ValueError(unemployment_response.get("message"))

unemployment_example = pd.DataFrame(
    unemployment_response["Results"]["series"][0]["data"]
)

unemployment_example.loc[
    unemployment_example["period"] == "M13",
    ["year", "value"]
].head()

,year,value
0,2023,5.3
13,2022,5.0
26,2021,9.2
39,2020,15.9
52,2019,4.6


In [38]:
# I list the New Jersey county codes as a reference; the API request already generates its own identifiers.
county_codes = ["34001", "34003", "34005", "34007", "34009", "34011", "34013",
                "34015", "34017", "34019", "34021", "34023", "34025", "34027",
                "34029", "34031", "34033", "34035", "34037", "34039", "34041"]

### 5.2 Load All Counties, Then Keep All Ten Years

In [39]:
# I combine the BLS annual-average records and check that all 210 county-years were returned.
# M13 is the published annual average, not a monthly observation.
unemployment_frames = []
for series in unemployment_response["Results"]["series"]:
    data = pd.DataFrame(series["data"])
    data = data.loc[data["period"] == "M13", ["year", "value"]].copy()
    data.columns = ["Date", "Unemployment_Rate"]
    data["Date"] = data["Date"] + "-01-01"
    data["County_FIPS"] = series["seriesID"][5:10]
    unemployment_frames.append(data)

unemployment_raw = pd.concat(unemployment_frames, ignore_index=True)
if len(unemployment_raw) != 210:
    raise ValueError("BLS did not return all 210 annual county-year records. Check the API response.")
unemployment_raw.head()


,Date,Unemployment_Rate,County_FIPS
0,2023-01-01,5.3,34001
1,2022-01-01,5.0,34001
2,2021-01-01,9.2,34001
3,2020-01-01,15.9,34001
4,2019-01-01,4.6,34001


In [40]:
# I extract the year and convert unemployment rates to numbers for merging and analysis.
unemployment_raw["Year"] = pd.to_datetime(unemployment_raw["Date"]).dt.year
unemployment_raw["Unemployment_Rate"] = pd.to_numeric(unemployment_raw["Unemployment_Rate"], errors="coerce")

In [41]:
# I keep the study years and the county, year, and unemployment columns I need.
unemployment = unemployment_raw.loc[
    unemployment_raw["Year"].isin(study_years),
    ["County_FIPS", "Year", "Unemployment_Rate"]
].copy()
unemployment.head()


,County_FIPS,Year,Unemployment_Rate
0,34001,2023,5.3
1,34001,2022,5.0
2,34001,2021,9.2
3,34001,2020,15.9
4,34001,2019,4.6


In [42]:
# I check that unemployment contains 210 rows and three columns.
unemployment.shape

(210, 3)

In [43]:
# I check unemployment data types before merging.
unemployment.dtypes

,0
County_FIPS,object
Year,int32
Unemployment_Rate,float64


In [44]:
# I check that unemployment includes all 21 counties in each study year.
unemployment["Year"].value_counts().sort_index()

,count
Year,
2014,21
2015,21
2016,21
2017,21
2018,21
2019,21
2020,21
2021,21
2022,21


In [45]:
# I check unemployment county-year keys for duplicates.
unemployment.duplicated(subset=merge_keys).sum()

np.int64(0)

In [46]:
# I check whether any unemployment values are missing.
unemployment.isna().sum()

,0
County_FIPS,0
Year,0
Unemployment_Rate,0


## 6. Prepare for the Second Merge

In [47]:
# I compare county and year coverage before adding unemployment to the combined table.
print("Alcohol/Census county codes:", alcohol_census["County_FIPS"].nunique())
print("BLS county codes:", unemployment["County_FIPS"].nunique())
print("Combined years:", sorted(alcohol_census["Year"].unique()))
print("BLS years:", sorted(unemployment["Year"].unique()))

Alcohol/Census county codes: 21
BLS county codes: 21
Combined years: [np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
BLS years: [np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023)]


# Merge 2: Add Unemployment for All Study Years

In [48]:
# I outer-merge unemployment with alcohol and Census using county and year, with one-to-one validation.
health_merged_check = pd.merge(
    alcohol_census,
    unemployment,
    how="outer",
    on=["County_FIPS", "Year"],
    indicator=True,
    validate="one_to_one"
)
health_merged_check.head()

,Year,County_FIPS,County,Alcohol_Prevalence,Alcohol_CI_Lower,Alcohol_CI_Upper,Poverty_Percent,Household_Income,Unemployment_Rate,_merge
0,2014,34001,Atlantic County,17.700000,17.200000,18.100000,15.1,54208,10.5,both
1,2015,34001,Atlantic County,17.307394,16.847267,17.757869,14.1,53296,9.5,both
2,2016,34001,Atlantic County,17.411235,16.917008,17.910904,14.2,55672,6.8,both
3,2017,34001,Atlantic County,17.004454,16.565697,17.468133,14.4,59309,6.5,both
4,2018,34001,Atlantic County,17.185672,16.541828,17.843667,12.8,60826,5.3,both


In [49]:
# I count matched and unmatched records to assess the second merge.
health_merged_check["_merge"].value_counts()

,count
_merge,
both,210
left_only,0
right_only,0


In [50]:
# I check that the second merge matched records in every study year.
pd.crosstab(health_merged_check["Year"], health_merged_check["_merge"])

_merge,both
Year,
2014,21
2015,21
2016,21
2017,21
2018,21
2019,21
2020,21
2021,21
2022,21


In [51]:
# I inspect individual county-year match statuses after adding unemployment.
health_merged_check[["County_FIPS", "Year", "_merge"]].sort_values(["Year", "County_FIPS"])

,County_FIPS,Year,_merge
0,34001,2014,both
10,34003,2014,both
20,34005,2014,both
30,34007,2014,both
40,34009,2014,both
...,...,...,...
169,34033,2023,both
179,34035,2023,both
189,34037,2023,both
199,34039,2023,both


In [52]:
# I display any unmatched records to check whether the second merge missed any keys.
unmatched = health_merged_check.loc[health_merged_check["_merge"] != "both"].copy()
unmatched

,Year,County_FIPS,County,Alcohol_Prevalence,Alcohol_CI_Lower,Alcohol_CI_Upper,Poverty_Percent,Household_Income,Unemployment_Rate,_merge


In [53]:
# I check that adding unemployment did not change the expected row count.
health_merged_check.shape

(210, 10)

In [54]:
# I check that the second merge did not create duplicate county-year records.
health_merged_check.duplicated(subset=merge_keys).sum()

np.int64(0)

In [55]:
# I check missing values after adding unemployment, including the existing alcohol gap.
health_merged_check.isna().sum()

,0
Year,0
County_FIPS,0
County,0
Alcohol_Prevalence,21
Alcohol_CI_Lower,21
Alcohol_CI_Upper,21
Poverty_Percent,0
Household_Income,0
Unemployment_Rate,0
_merge,0


### Interpretation of the Second Merge
All **210 county-year keys** match in the verified run because the combined dataset and BLS both retain the same 21 counties across all ten years. There are no new unmatched keys and no duplicated county-years.

The 21 missing 2019 alcohol estimates remain missing after this merge. This illustrates the distinction between a successful identifier match and an available numeric observation. Poverty, income and unemployment are available for every retained county-year.

In [56]:
# I remove the merge indicator and sort the final data by county and year.
health_merged = health_merged_check.drop(columns="_merge").copy()
health_merged = health_merged.sort_values(["County_FIPS", "Year"]).reset_index(drop=True)
health_merged.head()


,Year,County_FIPS,County,Alcohol_Prevalence,Alcohol_CI_Lower,Alcohol_CI_Upper,Poverty_Percent,Household_Income,Unemployment_Rate
0,2014,34001,Atlantic County,17.700000,17.200000,18.100000,15.1,54208,10.5
1,2015,34001,Atlantic County,17.307394,16.847267,17.757869,14.1,53296,9.5
2,2016,34001,Atlantic County,17.411235,16.917008,17.910904,14.2,55672,6.8
3,2017,34001,Atlantic County,17.004454,16.565697,17.468133,14.4,59309,6.5
4,2018,34001,Atlantic County,17.185672,16.541828,17.843667,12.8,60826,5.3


# Final Validation of the Full Ten-Year Dataset

In [57]:
# I check that my final merged table contains 210 rows and nine columns before analysis.
health_merged.shape

(210, 9)

In [58]:
# I confirm that every study year contains all 21 counties.
health_merged.groupby("Year")["County_FIPS"].nunique()

,County_FIPS
Year,
2014,21
2015,21
2016,21
2017,21
2018,21
2019,21
2020,21
2021,21
2022,21


In [59]:
# I confirm that every county has records for all ten study years.
health_merged.groupby("County")["Year"].nunique()

,Year
County,
Atlantic County,10
Bergen County,10
Burlington County,10
Camden County,10
Cape May County,10
Cumberland County,10
Essex County,10
Gloucester County,10
Hudson County,10


In [60]:
# I make a final check that each county-year appears only once.
health_merged.duplicated(subset=merge_keys).sum()

np.int64(0)

In [61]:
# I review missing values in the final dataset before analyzing it.
health_merged.isna().sum()

,0
Year,0
County_FIPS,0
County,0
Alcohol_Prevalence,21
Alcohol_CI_Lower,21
Alcohol_CI_Upper,21
Poverty_Percent,0
Household_Income,0
Unemployment_Rate,0


In [62]:
# I inspect records with missing alcohol estimates to confirm that they belong to 2019.
health_merged.loc[
    health_merged["Alcohol_Prevalence"].isna(),
    ["County", "Year", "Poverty_Percent", "Unemployment_Rate"]
]

,County,Year,Poverty_Percent,Unemployment_Rate
5,Atlantic County,2019,11.3,4.6
15,Bergen County,2019,5.7,2.8
25,Burlington County,2019,5.7,3.3
35,Camden County,2019,10.7,4.0
45,Cape May County,2019,9.8,7.2
55,Cumberland County,2019,13.8,5.7
65,Essex County,2019,13.8,4.1
75,Gloucester County,2019,7.5,3.6
85,Hudson County,2019,13.7,3.1
95,Hunterdon County,2019,4.0,2.7


### What the Final Panel Contains
The panel has 210 observations spanning all ten years, with 21 counties in each year. The alcohol comparison uses 189 available county-years; the 21 missing 2019 alcohol records remain in the final panel for economic analysis and coverage reporting.

Missing health values are excluded only from analyses that require alcohol prevalence. They are not removed from the full merged dataset.

# Full Analysis of the Merged Dataset

## 1. Label the Health Method Periods

In [63]:
# I label the alcohol method periods to document why estimates across years need careful interpretation.
health_merged["Method_Group"] = "Updated model / age-adjusted"
health_merged.loc[health_merged["Year"] <= 2017, "Method_Group"] = "Earlier model / not age-adjusted"
health_merged.loc[health_merged["Year"] == 2019, "Method_Group"] = "No new alcohol estimate"
health_merged.groupby(["Year", "Method_Group"])["Alcohol_Prevalence"].count()

,,Alcohol_Prevalence
Year,Method_Group,
2014,Earlier model / not age-adjusted,21
2015,Earlier model / not age-adjusted,21
2016,Earlier model / not age-adjusted,21
2017,Earlier model / not age-adjusted,21
2018,Updated model / age-adjusted,21
2019,No new alcohol estimate,0
2020,Updated model / age-adjusted,21
2021,Updated model / age-adjusted,21
2022,Updated model / age-adjusted,21


### Why Keep the Method Labels?
The source methodology and age adjustment changed. Yearly county comparisons can be described, but changes in the reported health values across method periods cannot be interpreted as real drinking changes. No continuous alcohol trend or before/after health effect is estimated.

## 2. Descriptive Statistics

In [64]:
# I summarize the available values, averages, and distributions of my main measures.
health_merged[["Alcohol_Prevalence", "Poverty_Percent", "Household_Income", "Unemployment_Rate"]].describe().round(2)

,Alcohol_Prevalence,Poverty_Percent,Household_Income,Unemployment_Rate
count,189.00,210.00,210.00,210.00
mean,18.41,9.78,83641.01,5.68
std,1.88,3.83,20373.68,2.27
min,14.09,3.90,45438.00,2.70
25%,17.09,6.62,66284.50,4.00
50%,18.13,9.25,80608.00,5.00
75%,19.40,12.55,98605.50,6.88
max,24.58,19.50,140242.00,15.90


### Interpretation of Overall Statistics
The alcohol count should be 189, while each economic variable has 210 observations. The pooled distribution describes the values included in the file, not an independent sample of 189 counties. The same 21 counties recur over years. Nominal income values across years are not used to infer real income growth.

In [65]:
# I summarize coverage and county averages separately for each study year.
year_summary = health_merged.groupby("Year").agg(
    Counties=("County_FIPS", "nunique"),
    Available_Alcohol=("Alcohol_Prevalence", "count"),
    Mean_County_Alcohol=("Alcohol_Prevalence", "mean"),
    Mean_County_Poverty=("Poverty_Percent", "mean"),
    Mean_County_Unemployment=("Unemployment_Rate", "mean")
).reset_index()
year_summary.round(2)

,Year,Counties,Available_Alcohol,Mean_County_Alcohol,Mean_County_Poverty,Mean_County_Unemployment
0,2014,21,21,17.93,10.94,7.12
1,2015,21,21,17.65,10.42,6.14
2,2016,21,21,17.84,10.33,5.31
3,2017,21,21,18.52,9.90,4.89
4,2018,21,21,18.80,9.29,4.33
5,2019,21,0,NaN,8.97,3.78
6,2020,21,21,20.16,9.22,9.67
7,2021,21,21,17.05,9.88,6.88
8,2022,21,21,19.28,9.30,4.12
9,2023,21,21,18.41,9.52,4.59


### Interpretation of Yearly Statistics
The table documents coverage in every study year and presents equally weighted county means. These means are not statewide population-weighted rates. The 2019 alcohol mean is missing, while 2019 economic statistics remain available. Alcohol means from different model periods are not treated as comparable trends.

In the verified data, mean county poverty is 10.94% in 2014 and 9.52% in 2023. Mean county unemployment is 7.12% in 2014, peaks at 9.67% in 2020, and is 4.59% in 2023. The economic patterns show why yearly alignment is useful: a historical average would hide the 2020 disruption.

## 3. County Alcohol Snapshots across the Study Period

In [66]:
# I arrange alcohol estimates into a county-by-year table for easier comparison.
alcohol_by_year = health_merged.pivot(index="County", columns="Year", values="Alcohol_Prevalence")
alcohol_by_year.round(1)

Year,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
County,,,,,,,,,,
Atlantic County,17.7,17.3,17.4,17.0,17.2,NaN,17.5,17.3,19.1,18.0
Bergen County,17.6,16.9,16.4,17.6,18.1,NaN,18.6,16.1,18.5,18.5
Burlington County,19.4,20.1,18.1,18.8,19.8,NaN,19.3,17.8,19.0,18.5
Camden County,17.7,17.7,17.0,18.1,17.5,NaN,20.0,17.7,19.2,18.4
Cape May County,18.1,18.3,18.8,18.1,21.4,NaN,23.3,17.5,19.9,21.2
Cumberland County,15.9,16.4,17.2,17.1,18.5,NaN,18.6,17.1,17.5,16.1
Essex County,16.4,15.5,15.4,16.2,17.3,NaN,17.3,15.5,16.9,16.6
Gloucester County,20.3,18.8,20.4,20.9,18.9,NaN,22.4,18.4,19.5,18.9
Hudson County,18.6,17.4,16.9,19.2,18.1,NaN,20.4,16.9,19.5,17.2


In [67]:
# I plot a heatmap of county alcohol estimates, keeping the missing 2019 values visible.
alcohol_heatmap = px.imshow(alcohol_by_year, aspect="auto",
    labels={"x": "Observation year", "y": "County", "color": "Excessive drinking (%)"},
    title="Reported County Alcohol Snapshots, 2014-2023 (Methods Change; 2019 Missing)", height=750)
alcohol_heatmap.update_layout(template="plotly_white", plot_bgcolor="#eeeeee")
alcohol_heatmap.show()

### Interpretation of the County-Year Heatmap
The blank 2019 column is a source coverage gap. Colors allow within-year county comparisons. A color shift over years is not by itself evidence of a real alcohol trend, because modeling and age adjustment differ across releases. This heatmap makes the complete study scope and missing observations visible.

For the 2023 snapshot, county excessive-drinking estimates range from 15.8% to 21.6%, with an equally weighted county mean of 18.41%. These are excessive-drinking estimates, not the earlier notebook's narrower binge-drinking estimates.

## 4. Compare Alcohol with Poverty within Each Year

In [68]:
# I create a complete-data subset for alcohol comparisons while keeping the full merged dataset.
analysis_data = health_merged.dropna(subset=["Alcohol_Prevalence", "Poverty_Percent", "Unemployment_Rate"]).copy()
analysis_data.shape

(189, 10)

In [69]:
# I compare poverty and alcohol estimates across counties within each available year.
poverty_figure = px.scatter(analysis_data, x="Poverty_Percent", y="Alcohol_Prevalence",
    hover_name="County", facet_col="Year", facet_col_wrap=3, height=850,
    title="County Poverty and Excessive Drinking: Separate Observation-Year Comparisons",
    labels={"Poverty_Percent": "Poverty (%)", "Alcohol_Prevalence": "Excessive drinking (%)"})
poverty_figure.update_layout(template="plotly_white")
poverty_figure.show()

### Interpretation of the Poverty Comparisons
Each panel compares counties measured in the same observation year. This avoids mixing different years in one scatterplot. These are modeled health estimates, and poverty contributes to their estimation, so associations with poverty are descriptive rather than independent causal evidence.

The observed poverty-alcohol correlation is negative in every available year, ranging from approximately -0.59 to -0.37. In these modeled county snapshots, counties with more poverty do not generally have higher estimated excessive drinking. This should not be interpreted as poverty protecting individuals from alcohol-related harm.

## 5. Compare Alcohol with Unemployment within Each Year

In [70]:
# I compare unemployment and alcohol estimates across counties within each available year.
unemployment_figure = px.scatter(analysis_data, x="Unemployment_Rate", y="Alcohol_Prevalence",
    hover_name="County", facet_col="Year", facet_col_wrap=3, height=850,
    title="County Unemployment and Excessive Drinking: Separate Observation-Year Comparisons",
    labels={"Unemployment_Rate": "Unemployment (%)", "Alcohol_Prevalence": "Excessive drinking (%)"})
unemployment_figure.update_layout(template="plotly_white")
unemployment_figure.show()

### Interpretation of the Unemployment Comparisons
The second merge allows each county's alcohol estimate to be compared with its annual unemployment rate in that same year. The panel structure is preserved; no historical averages substitute for yearly observations. Differences in these plots do not establish that unemployment changes drinking behavior.

Year-specific alcohol-unemployment correlations range from about -0.29 to 0.04. These are mostly weak relationships; the 2023 correlation is approximately 0.01. Unemployment alone does not show a strong descriptive relationship with the county alcohol estimates.

## 6. Year-Specific Correlation Summary

In [71]:
# I calculate separate yearly correlations to describe alcohol's relationships with poverty, unemployment, and income.
correlation_rows = []
for year in sorted(analysis_data["Year"].unique()):
    data = analysis_data.loc[analysis_data["Year"] == year]
    correlation_rows.append({
        "Year": year,
        "Counties": len(data),
        "Alcohol_Poverty_r": data["Alcohol_Prevalence"].corr(data["Poverty_Percent"]),
        "Alcohol_Unemployment_r": data["Alcohol_Prevalence"].corr(data["Unemployment_Rate"]),
        "Alcohol_Income_r": data["Alcohol_Prevalence"].corr(data["Household_Income"])
    })
correlations = pd.DataFrame(correlation_rows)
correlations.round(2)

,Year,Counties,Alcohol_Poverty_r,Alcohol_Unemployment_r,Alcohol_Income_r
0,2014,21,-0.37,-0.25,0.44
1,2015,21,-0.48,-0.17,0.38
2,2016,21,-0.59,-0.13,0.46
3,2017,21,-0.48,-0.29,0.36
4,2018,21,-0.57,0.04,0.39
5,2020,21,-0.48,-0.16,0.25
6,2021,21,-0.37,-0.18,0.25
7,2022,21,-0.49,-0.17,0.42
8,2023,21,-0.52,0.01,0.26


### Interpretation of the Correlation Summary
The table gives a separate descriptive correlation for each available alcohol observation year. Income is compared within years, so a single nominal-income level is not used to compare purchasing power across years. Each correlation uses 21 counties and ignores estimate uncertainty, age composition and spatial dependence. No formal significance claim or causal effect is inferred.

Household income is positively associated with alcohol estimates within every available year (approximately 0.25 to 0.46), while poverty is negatively associated. Unemployment associations are weaker. These patterns do not support a simple hypothesis that greater county economic disadvantage always corresponds to higher reported excessive drinking. The consistent directions motivate questions about demographic composition and modeled-data inputs rather than causal claims.

## 7. Recode Alcohol Relative to the County Median in Each Year

In [72]:
# I group counties by their own year's alcohol median and keep missing estimates in a separate category.
health_merged["Year_Alcohol_Median"] = health_merged.groupby("Year")["Alcohol_Prevalence"].transform("median")
health_merged["Alcohol_Category"] = "Missing Alcohol"
health_merged.loc[health_merged["Alcohol_Prevalence"] < health_merged["Year_Alcohol_Median"],
                  "Alcohol_Category"] = "Below Year Median"
health_merged.loc[health_merged["Alcohol_Prevalence"] >= health_merged["Year_Alcohol_Median"],
                  "Alcohol_Category"] = "At or Above Year Median"
health_merged.groupby(["Year", "Alcohol_Category"]).size()

Year  Alcohol_Category       
2014  At or Above Year Median    12
      Below Year Median           9
2015  At or Above Year Median    11
      Below Year Median          10
2016  At or Above Year Median    11
      Below Year Median          10
2017  At or Above Year Median    11
      Below Year Median          10
2018  At or Above Year Median    11
      Below Year Median          10
2019  Missing Alcohol            21
2020  At or Above Year Median    11
      Below Year Median          10
2021  At or Above Year Median    11
      Below Year Median          10
2022  At or Above Year Median    11
      Below Year Median          10
2023  At or Above Year Median    11
      Below Year Median          10
dtype: int64

In [73]:
# I compare average economic conditions across the alcohol groups within each year.
group_summary = health_merged.groupby(["Year", "Alcohol_Category"]).agg(
    Observations=("County_FIPS", "count"),
    Mean_Alcohol=("Alcohol_Prevalence", "mean"),
    Mean_Poverty=("Poverty_Percent", "mean"),
    Mean_Unemployment=("Unemployment_Rate", "mean"),
    Mean_Household_Income=("Household_Income", "mean")
).reset_index()
group_summary.round(2)

,Year,Alcohol_Category,Observations,Mean_Alcohol,Mean_Poverty,Mean_Unemployment,Mean_Household_Income
0,2014,At or Above Year Median,12,18.83,10.69,7.19,74493.25
1,2014,Below Year Median,9,16.73,11.28,7.02,68991.33
2,2015,At or Above Year Median,11,18.67,9.52,6.02,75310.18
3,2015,Below Year Median,10,16.52,11.42,6.27,69738.50
4,2016,At or Above Year Median,11,18.88,7.68,4.98,85021.09
5,2016,Below Year Median,10,16.70,13.24,5.67,65319.00
6,2017,At or Above Year Median,11,19.70,8.01,4.64,85868.09
7,2017,Below Year Median,10,17.21,11.97,5.16,72930.50
8,2018,At or Above Year Median,11,20.18,7.05,4.25,88591.55
9,2018,Below Year Median,10,17.28,11.74,4.43,73648.50


### Interpretation of Grouped Statistics
The categories compare counties with their same-year median, not a medical threshold or one threshold pooled across methods. The 2019 rows are explicitly labeled Missing Alcohol, so missing values are not accidentally classified as lower prevalence. Economic group averages remain descriptive county context.

## 8. Observe Economic Conditions across All Ten Years

In [74]:
# I plot county poverty across all ten years to show changes in economic conditions.
poverty_history = px.line(health_merged, x="Year", y="Poverty_Percent", color="County",
    title="Observed County Poverty, New Jersey, 2014-2023",
    labels={"Poverty_Percent": "All-age poverty (%)"})
poverty_history.update_layout(template="plotly_white")
poverty_history.show()

### Interpretation of Poverty over Time
The Census annual county series provides genuine time-indexed economic observations for all ten years, including 2019. The trajectory describes estimated poverty changes; it does not identify the causes of those changes or a response in county drinking.

In [75]:
# I plot county unemployment across all ten years to show its economic history.
unemployment_history = px.line(health_merged, x="Year", y="Unemployment_Rate", color="County",
    title="Observed County Unemployment, New Jersey, 2014-2023",
    labels={"Unemployment_Rate": "Annual unemployment (%)"})
unemployment_history.update_layout(template="plotly_white")
unemployment_history.show()

### Interpretation of Unemployment over Time
The annual county unemployment series shows the 2020 disruption and subsequent recovery. Unlike the alcohol archive, these economic series are not labeled as carried-forward county health estimates. The plot provides context rather than a causal pandemic analysis.

# Limitations

1. The ten-year panel includes all study years, but New Jersey has no new 2019 alcohol estimate. That year's alcohol fields remain missing, while its economic observations are retained.
2. Excessive drinking is broader than the binge drinking measure in PS2. The notebook does not combine these measures as if they were identical.
3. Alcohol values are modeled snapshots, and methods/age adjustment changed. Cross-release differences should not be interpreted as real county drinking trends.
4. Model inputs include socioeconomic information, so observed economic correlations may partly reflect the estimation method.
5. The same 21 counties recur over time; pooled rows are not independent counties. Year-specific correlations are descriptive and do not model spatial dependence or uncertainty.
6. Income is nominal dollars. It is compared within observation years and is not used to claim real income growth.
7. County associations do not identify an individual-level relationship or causal effect.
8. Source files may be revised or temporarily unavailable. The documented URLs and observation years identify the sources used. All datasets are requested online.


# Final Summary

The project now merges on County_FIPS and Year across the full 2014-2023 study period. Dataset 1 is county excessive drinking; Merge 1 adds annual Census income and poverty; Merge 2 adds annual BLS unemployment. The first merge explains the missing 2019 alcohol keys, and the second merge checks all 210 county-year keys.

The final analysis retains all ten study years, distinguishes missing estimates from failed identifier matches, compares alcohol and economic conditions within actual observation years, and describes complete economic histories. No single-year merge, prior economic average, repeated 2023 health value or invented 2019 alcohol estimate is used.

The final verified panel has 210 county-year records, no duplicated keys, complete annual economic data and 189 available alcohol estimates. The source gap is confined to 2019. Across available years, higher-income counties generally have higher estimated excessive drinking, poverty has a negative descriptive association, and unemployment has a weak association. These results answer the same-year county comparison question while respecting the limitations of modeled health estimates and method changes.

###AI Assistance

ChatGPT (OpenAI) was used to help identify relevant data sources and API endpoints, organize the loading of data across multiple years, and assist with combining the yearly datasets. The original data sources are County Health Rankings & Roadmaps using CDC BRFSS estimates, the U.S. Census Bureau, and the U.S. Bureau of Labor Statistics.